# Chapter 13 — Random walks

Companion notebook to `docs/models/ch13_random_walk.md` and the Streamlit page `app/pages/13_Random_Walks.py`.

Four ideas, one notebook:
1. The **√t rule** — distance grows as the square root of time.
2. The **return-time paradox** — certain return, infinite expected wait.
3. **Pólya's theorem** — drunk men come home, drunk birds may not.
4. The **arcsine law** — fair contests look one-sided.

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import matplotlib.pyplot as plt
import numpy as np

from modelthinker.dynamics.random_walk import (
    arcsine_fractions,
    dimension_recurrence,
    many_walks,
    return_times,
    rms_displacement,
    theoretical_return_survival,
)

## 1. The √t rule — distance grows as the square root of time

In [ ]:
r = many_walks(n_walks=1000, steps=2000, seed=42)
t = np.arange(r.positions.shape[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i in range(200):
    axes[0].plot(t, r.positions[i], color='#4C6EF5', alpha=0.15, lw=0.6)
axes[0].fill_between(t, -np.sqrt(t), np.sqrt(t), color='#F76707', alpha=0.2, label=r'$\pm\sqrt{t}$')
axes[0].set(title='200 walks with the √t envelope', xlabel='step', ylabel='position')
axes[0].legend()

rms = rms_displacement(r)
axes[1].loglog(t[1:], rms[1:], color='#4C6EF5', lw=2, label='empirical RMS')
axes[1].loglog(t[1:], np.sqrt(t[1:]), 'r--', lw=2, label=r'theory $\sqrt{t}$')
axes[1].set(title='RMS displacement — a straight line on log-log', xlabel='t (log)', ylabel='RMS (log)')
axes[1].legend(); axes[1].grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

## 2. The return-time paradox

In [ ]:
rt = return_times(n_walks=10_000, max_steps=20_000, seed=1)
ret = rt.returned_times

print(f'Returned within budget:  {100 * rt.return_fraction:.1f}%')
print(f'Fraction returning at t=2:  {(ret == 2).sum() / len(ret):.3f}   (theory 0.500)')
print(f'Median return time:  {int(np.median(ret))} steps')
print(f'99th percentile:     {int(np.percentile(ret, 99))} steps')
print(f'Empirical mean:      {ret.mean():.1f} steps   (theoretical: infinite)')

ts = np.geomspace(2, rt.max_steps, 60)
emp_surv = np.array([(ret > ti).sum() / rt.n_walks + (rt.times == 0).sum() / rt.n_walks for ti in ts])
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(ts, emp_surv, 'o-', color='#4C6EF5', label='empirical P(T > t)')
ax.loglog(ts, theoretical_return_survival(ts), 'r--', lw=2, label=r'theory $\sqrt{2/(\pi t)}$')
ax.set(xlabel='t (log)', ylabel='P(T > t) (log)', title='Return-time survival — power-law tail')
ax.legend(); ax.grid(True, which='both', alpha=0.3); plt.show()

## 3. Pólya's theorem

In [ ]:
dims = [1, 2, 3, 4]
theory = [1.0, 1.0, 0.3405, 0.193]
res = [dimension_recurrence(dim=d, n_walks=2000, max_steps=20_000, seed=10+d) for d in dims]
empirical = [r.return_fraction for r in res]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([f'{d}-D' for d in dims], empirical, color='#4C6EF5', alpha=0.85, label='empirical (20k steps)')
ax.plot([f'{d}-D' for d in dims], theory, 'D', color='#E03131', markersize=12, label='theory limit')
ax.set(ylabel='fraction returning to origin', title="Pólya's theorem in miniature", ylim=(0, 1.05))
for i, v in enumerate(empirical):
    ax.text(i, v + 0.02, f'{100*v:.1f}%', ha='center')
ax.legend(); plt.show()

## 4. The arcsine law

In [ ]:
fracs = arcsine_fractions(n_walks=10_000, steps=500, seed=99)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(fracs, bins=25, density=True, color='#4C6EF5', alpha=0.7, label='empirical')
xs = np.linspace(0.01, 0.99, 200)
ax.plot(xs, 1 / (np.pi * np.sqrt(xs * (1 - xs))), 'r-', lw=2, label='arcsine density')
ax.set(xlabel='fraction of time position > 0', ylabel='density',
       title='Fair walks look one-sided')
ax.legend(); plt.show()

print(f'Fraction near extremes (<10% or >90%): {((fracs<0.1)|(fracs>0.9)).mean():.3f}')
print(f'Fraction near middle (40-60%):         {((fracs>0.4)&(fracs<0.6)).mean():.3f}')

## 5. What-ifs to try

1. Add a tiny drift (`many_walks(..., step_type='normal', drift=0.05)`) and re-run tab 1 — the walker still looks noisy but climbs steadily. This is the 'trend vs noise' problem in one picture.
2. Run `dimension_recurrence(dim=2, n_walks=1000, max_steps=200_000)` — the 2-D fraction climbs, but glacially (1/log t).
3. Simulate 500 mutual funds as normal walks of 60 months. How many will look like 'top-quartile performers' every year purely by chance?